In [1]:
# Loading libraries
import os
from dotenv import load_dotenv

import numpy as np 
import warnings
warnings.filterwarnings("ignore")
load_dotenv()

True

In [ ]:
from langchain_community.vectorstores import FAISS

Data Ingestion and Processing

In [ ]:
from langchain_core.documents import Document
sample_documents = [
    Document(
        page_content="""
        MachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data without being explicitly programmed. Instead of following fixed rules, machine learning systems improve their performance over time as they are exposed to more information. It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled data. Machine learning is widely used in applications like recommendation systems, fraud detection, predictive analytics, and medical diagnosis, making it a cornerstone of modern AI.",
        """,
        metadata={"source":"ML Introduction","page":1,"topic":"ML"}
    ),
    Document(
        page_content="""
        DeepLearning: "Deep learning is a specialized subset of machine learning that uses multi-layered artificial neural networks to automatically learn complex features and representations from raw data. Unlike traditional machine learning, which often requires manual feature engineering, deep learning models can process unstructured data such as images, audio, and text directly, extracting hierarchical features through successive layers. These networks, often containing millions of parameters, are trained using large datasets and powerful computational resources, enabling breakthroughs in areas like computer vision, natural language processing, autonomous driving, and speech recognition. Deep learning has revolutionized AI by achieving human-level or even superhuman performance in tasks that were previously considered extremely difficult for machines.",
        """,
        metadata={'source':"DL Introduction","page":1,"topic":"DL"}
    ),
    Document(
        page_content="""
        NeuralNetworks: "Neural networks are computational models inspired by the structure and functioning of the human brain. They consist of interconnected nodes, called neurons, organized into layers: an input layer, one or more hidden layers, and an output layer. Each neuron applies mathematical transformations to its inputs and passes the result forward, allowing the network to learn complex, non-linear relationships in data. Neural networks can be shallow, with only a few layers, or deep, with many hidden layers, forming the basis of deep learning. Variants such as convolutional neural networks (CNNs) excel at image recognition by capturing spatial hierarchies, while recurrent neural networks (RNNs) are designed to handle sequential data like text or time series. Neural networks are fundamental to modern AI, enabling systems to recognize patterns, make predictions, and adapt to new information."
        """,
        metadata={'source':"NN Introduction","page":1,"topic":"NN"}
    )
]

In [24]:
print(sample_documents)

NameError: name 'sample_documents' is not defined

In [23]:
###Test splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitting=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
    separators=[" "]
)

In [11]:
chunks=text_splitting.split_documents(sample_documents)

NameError: name 'sample_documents' is not defined

In [ ]:
chunks

[Document(metadata={'source': 'ML Introduction', 'page': 1, 'topic': 'ML'}, page_content='MachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data'),
 Document(metadata={'source': 'ML Introduction', 'page': 1, 'topic': 'ML'}, page_content='from data without being explicitly programmed. Instead of following fixed rules, machine learning systems improve their performance over time as they are exposed to more information. It encompasses'),
 Document(metadata={'source': 'ML Introduction', 'page': 1, 'topic': 'ML'}, page_content='It encompasses techniques such as supervised learning, where models are trained on labeled data to make predictions, and unsupervised learning, where systems discover hidden structures in unlabeled'),
 Document(metadata={'source': 'ML Introduction', 'page': 1, 'topic': 'ML'}, page_content='in unlabeled data. Machine learning is widely u

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2'
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)

NameError: name 'FAISS' is not defined

In [ ]:
query = "What is Machine Learning"
results = vectorstore.similarity_search(query)
print(results[0])

page_content='MachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data' metadata={'source': 'ML Introduction', 'page': 1, 'topic': 'ML'}


In [ ]:
#Compare the vectors using cosine similarity
def compare_embeddings(text1:str ,text2:str):
    """
    Compare Semantic similarity of 2 texts using embddings
    """
    emb1=np.array(embeddings.embed_query(text1))
    emb2=np.array(embeddings.embed_query(text2))
    #calculte the similarity search
    similarity=np.dot(emb1,emb2)/(np.linalg.norm(emb1)*np.linalg.norm(emb2))
    return similarity

In [ ]:
print("Semantic similarity Examples: ")
print(f"'AI' vs 'Artificial Intelligence': {compare_embeddings('AI', 'Artificial Intelligence'):.3f}")

Semantic similarity Examples: 
'AI' vs 'Artificial Intelligence': 0.791


Creation of FAISS Vector Store

In [ ]:
vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector stroe  created with {vectorstore.index.ntotal} vector")

Vector stroe  created with 16 vector


In [ ]:
vectorstore.save_local('faiss_local')

In [12]:
#Now Loading the vector_store
loaded_vectorstore=FAISS.load_local(
    "faiss_local",
    embeddings,
    allow_dangerous_deserialization=True
)
print(f"Loaeded vector store contains {loaded_vectorstore.index.ntotal} vector ")

NameError: name 'FAISS' is not defined

In [13]:
query="What is neural Network?"
result=vectorstore.similarity_search(query,k=3)
result

NameError: name 'vectorstore' is not defined

In [14]:
filter_dict={"topic":"ML"}
filter_results=vectorstore.similarity_search(
    query,
    k=3,
    filter=filter_dict
)
print(filter_results)

NameError: name 'vectorstore' is not defined

In [15]:
len(filter_results)

NameError: name 'filter_results' is not defined

In [16]:
###Build RAG CHAIN WITH LCEL

In [17]:
from langchain_groq import ChatGroq
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
llm=ChatGroq(model="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001ECB3762A20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001ECB389AE70>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [18]:
llm.invoke("Hi")

AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 36, 'total_tokens': 44, 'completion_time': 0.006815675, 'completion_tokens_details': None, 'prompt_time': 0.002600098, 'prompt_tokens_details': None, 'queue_time': 0.052347672, 'total_time': 0.009415773}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b5b7b-db12-70d3-9f6a-7ffdfec2784f-0', usage_metadata={'input_tokens': 36, 'output_tokens': 8, 'total_tokens': 44})

In [19]:
#Simple RAG Chain with LCEL
from langchain_core.prompts import ChatPromptTemplate
single_prompt=ChatPromptTemplate.from_template(
    """
Answer the question Based on follwing Context:
Context:{context}
Question:{question}
Answer:

"""
)


In [20]:
vectorstore

NameError: name 'vectorstore' is not defined

In [ ]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000173A1206900>, search_kwargs={'k': 3})

In [ ]:
from typing import List
def format_docs(docs:List[Document])->str:
    """Format Documents for insertion into prompt"""
    formatted=[]
    for i,doc in enumerate(docs):
        source=doc.metadata.get('source',"unknown")
        formatted.append(f"Document {i+1} (Source:{source}):\n{doc.page_content}")
    return "\n\n".join(formatted)



In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
single_rag_chain=(
    {"context":retriever| format_docs,"question":RunnablePassthrough()}
    |single_prompt
    |llm
    | StrOutputParser
    
)

In [ ]:
#Conversational prompt
from langchain_core.prompts import MessagesPlaceholder
conversational_prompt=ChatPromptTemplate.from_messages([
    ("system","You are a helpful AI Assistant. Use the provided context to answer questions"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human","Context:{context}\n\n Question:{input}"),

])

In [ ]:
single_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000173A1206900>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question Based on follwing Context:\nContext:{context}\nQuestion:{question}\nAnswer:\n\n'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000173CFAC0DD0>, async_cl

In [ ]:
def create_conversational_rag():
    """Create a conversational RAG Chain with memory"""
    return(
        RunnablePassthrough.assign(
            content=lambda x: format_docs(retriever.invoke(x['input']))

        )
        | conversational_prompt
        | llm 
        | StrOutputParser()
    )

In [ ]:
print("Modern RAG CHAINS Created successfully")
print("Available Chains")
print("-Simple_rag_chain:Basic Q&A")
print("-Conversational_rag:Maintains Conversation history")
print("-Streaming_rag_chain:support token streaming")

Modern RAG CHAINS Created successfully
Available Chains
-Simple_rag_chain:Basic Q&A
-Conversational_rag:Maintains Conversation history
-Streaming_rag_chain:support token streaming


In [ ]:
def test_rag_chains(question:str):
    """Test all R&G chain variants"""
    print(f"Question:{question}")
    print("="*80)

    #1 Simple RAG
    print("\n1 simple rag chain:")
    answer=single_rag_chain.invoke(question)
    print(answer)

In [ ]:
test_rag_chains("What is the difference between AI and Machine learning?")

Question:What is the difference between AI and Machine learning?

1 simple rag chain:


TypeError: BaseModel.__init__() takes 1 positional argument but 2 were given

In [ ]:
prompt_val = single_prompt.invoke(result)
prompt_val

ChatPromptValue(messages=[HumanMessage(content='\nAnswer the question Based on follwing Context:\nContext:Document 1 (Source:ML Introduction):\nMachineLearning: "Machine learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data\n\nDocument 2 (Source:ML Introduction):\nin unlabeled data. Machine learning is widely used in applications like recommendation systems, fraud detection, predictive analytics, and medical diagnosis, making it a cornerstone of modern AI.",\n\nDocument 3 (Source:ML Introduction):\nfrom data without being explicitly programmed. Instead of following fixed rules, machine learning systems improve their performance over time as they are exposed to more information. It encompasses\nQuestion:What is Machine Learning\nAnswer:\n\n', additional_kwargs={}, response_metadata={})])

In [ ]:
ai_msg = llm.invoke(prompt_val)
ai_msg

AIMessage(content='Based on the provided context, Machine Learning is a field of artificial intelligence that focuses on creating algorithms and models that enable computers to learn patterns and relationships from data, particularly from unlabeled data.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 179, 'total_tokens': 218, 'completion_time': 0.054950937, 'completion_tokens_details': None, 'prompt_time': 0.011429185, 'prompt_tokens_details': None, 'queue_time': 0.049775085, 'total_time': 0.066380122}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b5a63-fe07-70e1-947e-aa942398ef7b-0', usage_metadata={'input_tokens': 179, 'output_tokens': 39, 'total_tokens': 218})

In [ ]:
from langchain_core.runnables import RunnableParallel
parallel = RunnableParallel(context= retriever | format_docs, question= RunnablePassthrough())
result = parallel.invoke("What is Machine Learning")
result

NameError: name 'retriever' is not defined